In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import pandas as pd
import time

# 1. Setup Device & Reproducibility
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

# 2. Data Loading (Fixed Hyperparameters)
BATCH_SIZE = 64
EPOCHS = 5
LEARNING_RATE = 0.01

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Using a subset (e.g., 10,000 images) to make training fast and distinct across optimizers
train_subset = Subset(train_dataset, range(10000))
test_subset = Subset(test_dataset, range(2000))

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False)

# Full batch loader specifically for "Batch Gradient Descent" simulation
full_train_loader = DataLoader(train_subset, batch_size=len(train_subset), shuffle=False)


In [7]:
# 3. Fixed Model Architecture
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

In [8]:
# 4. Training and Validation Function
def train_and_evaluate(optimizer_name, custom_optimizer=None, is_batch_gd=False):
    model = SimpleCNN().to(device)
    criterion = nn.CrossEntropyLoss()

    # Initialize appropriate optimizer
    if is_batch_gd:
        # Batch GD uses standard SGD but updates once per epoch over the entire dataset
        optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)
    else:
        optimizer = custom_optimizer(model.parameters())

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    start_time = time.time()

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        # Select active loader based on algorithm type
        current_loader = full_train_loader if is_batch_gd else train_loader

        for images, labels in current_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        epoch_train_loss = running_loss / total_train
        epoch_train_acc = (correct_train / total_train) * 100

        # Validation Phase
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs.data, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()

        epoch_val_loss = val_loss / total_val
        epoch_val_acc = (correct_val / total_val) * 100

        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_acc'].append(epoch_val_acc)

    execution_time = time.time() - start_time
    print(f"Finished {optimizer_name} in {execution_time:.2f}s | Final Val Acc: {epoch_val_acc:.2f}%")

    return history, epoch_val_acc, epoch_val_loss, execution_time

In [9]:
# 5. Execute Experiments
optimizers_config = {
    'Batch GD': (None, True),
    'SGD (Stochastic)': (lambda params: optim.SGD(params, lr=LEARNING_RATE), False), # Using small batch mimics stochastic trends cleanly
    'Mini-Batch SGD': (lambda params: optim.SGD(params, lr=LEARNING_RATE), False),
    'SGD + Momentum': (lambda params: optim.SGD(params, lr=LEARNING_RATE, momentum=0.9), False),
    'Nesterov Momentum': (lambda params: optim.SGD(params, lr=LEARNING_RATE, momentum=0.9, nesterov=True), False),
    'RMSprop': (lambda params: optim.RMSprop(params, lr=0.001), False),
    'Adam': (lambda params: optim.Adam(params, lr=0.001), False)
}

results = {}
summary_table = []

for name, (opt_fn, is_bgd) in optimizers_config.items():
    # To truly simulate pure single-sample SGD, we change batch size dynamically for that run
    if name == 'SGD (Stochastic)':
        train_loader = DataLoader(train_subset, batch_size=1, shuffle=True)
    elif name == 'Mini-Batch SGD':
        train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)

    hist, final_acc, final_loss, duration = train_and_evaluate(name, opt_fn, is_batch_gd=is_bgd)
    results[name] = hist
    summary_table.append({
        'Optimizer': name,
        'Final Val Accuracy (%)': round(final_acc, 2),
        'Final Val Loss': round(final_loss, 4),
        'Execution Time (s)': round(duration, 2)
    })


Finished Batch GD in 61.35s | Final Val Acc: 14.65%
Finished SGD (Stochastic) in 121.88s | Final Val Acc: 96.95%
Finished Mini-Batch SGD in 40.62s | Final Val Acc: 89.15%
Finished SGD + Momentum in 40.41s | Final Val Acc: 96.90%
Finished Nesterov Momentum in 41.10s | Final Val Acc: 96.90%
Finished RMSprop in 40.84s | Final Val Acc: 97.30%
Finished Adam in 42.21s | Final Val Acc: 97.20%
